# Day 3. Fine-Tune, Evaluate Honestly, and Ship

The capstone. You train a left-right classifier on a small labeled set, prove it beats the
baseline you built yesterday, apply it over a whole corpus, and save it for reuse. Switch to a
GPU runtime now with Runtime then Change runtime type then T4 GPU.

## Load the workshop helpers

One line fetches the helper functions we use across the workshop. Open
`workshop_utils.py` in the file browser on the left if you want to read them.

In [ ]:
!wget -q -O workshop_utils.py https://raw.githubusercontent.com/jacqpark/instats-python-workshop/main/workshop_utils.py
from workshop_utils import pull_manifesto
print('helpers loaded')

## The training loop, in ten lines

Training is not magic. It is a loop. For each batch you predict, measure how wrong you are, and
nudge the weights to be a little less wrong. Roughly this.

```
for epoch in range(epochs):
    for batch in data:
        preds = model(batch.x)
        loss = how_wrong(preds, batch.y)
        loss.backward()        # which way to nudge each weight
        optimizer.step()       # take the nudge
        optimizer.zero_grad()  # reset for the next batch
```

SetFit wraps this loop for us and works from very few labeled examples, which is why everyone can
finish it in the session.

## Load and make a few-shot split

Pull the corpus, collapse to left and right, then take a small balanced training set and a larger
held-out test set. Few-shot means few labels, which is the realistic case for hand-coded data.

In [ ]:
from google.colab import userdata
import pandas as pd

df = pull_manifesto(userdata.get('MANIFESTO_KEY'))
df = df[df['rile'].notna()].reset_index(drop=True)
# One country keeps the vocabulary consistent, which is what a small
# training set needs. All four at once costs about 10 kappa points.
df = df[df['countryname'] == 'United Kingdom'].reset_index(drop=True)
df['label'] = df['rile']

n_per_class = 64
train = df.groupby('label').sample(n=n_per_class, random_state=42)
test = df.drop(train.index).groupby('label').sample(n=200, random_state=42)
print('train', len(train), 'test', len(test))

## Fine-tune with SetFit

This trains in minutes on a free T4. It is the capstone everyone completes.

In [ ]:
from setfit import SetFitModel, Trainer, TrainingArguments
from datasets import Dataset

model = SetFitModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
train_ds = Dataset.from_dict({'text': train['text'].tolist(), 'label': train['label'].tolist()})
trainer = Trainer(model=model, args=TrainingArguments(batch_size=16, num_epochs=1),
                  train_dataset=train_ds)
trainer.train()
print('trained')

## First look at predictions

Run the model on a few held-out sentences and sanity-check what it learned.

In [ ]:
for txt in test['text'].head(4):
    print(model.predict([txt])[0], '<-', txt[:70])

## Did it help, against the baseline you built

Yesterday's TF-IDF baseline is the bar. Score both on the held-out set with Cohen's kappa and
macro F1. On real manifesto text the fine-tune beats the bag of words, which is the whole reason
to fine-tune. On easier tasks the baseline can be enough, so always check.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix

vec = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=20000)
clf = LogisticRegression(max_iter=1000)
clf.fit(vec.fit_transform(train['text']), train['label'])
base_pred = clf.predict(vec.transform(test['text']))
setfit_pred = model.predict(test['text'].tolist())
y = test['label'].tolist()

for name, pred in [('tf-idf baseline', base_pred), ('setfit fine-tune', setfit_pred)]:
    print(f'{name:18s} kappa={cohen_kappa_score(y,pred):.2f}  macroF1={f1_score(y,pred,average="macro"):.2f}')
print('confusion (setfit), rows true left/right:')
print(confusion_matrix(y, setfit_pred, labels=['left','right']))

## Run your model over a large corpus

The payoff. Apply the fine-tuned model to every sentence, then aggregate to a left-right score per
party and year. That reconstructs a scaling measure from raw text, at a scale hand-coding cannot
reach. Party and year come from the Manifesto pull.

In [ ]:
import numpy as np
sample = df.sample(min(2000, len(df)), random_state=0).copy()
sample['pred'] = model.predict(sample['text'].tolist())
sample['is_right'] = (sample['pred'] == 'right').astype(int)
scale = sample.groupby(['party','year'])['is_right'].mean().reset_index(name='right_share')
print(scale.head(10))

## Ship it

Save the model, reload it to prove the save worked, then push it to the Hub so others can use it.
The push needs a Hugging Face token, added to Colab secrets like your Manifesto key.

In [ ]:
model.save_pretrained('left_right_setfit')
reloaded = SetFitModel.from_pretrained('left_right_setfit')
print('reloaded predicts:', reloaded.predict(['Expand public healthcare for all.'])[0])

# from huggingface_hub import login
# login(userdata.get('HF_TOKEN'))
# model.push_to_hub('your-username/manifesto-left-right-setfit')

## Write a model card

A model card is the readme that ships with a model. Say what it does, the data, the score, and
the seed, so your measurement is reproducible.

```
# Manifesto left-right SetFit classifier
Fine-tuned all-MiniLM-L6-v2 with SetFit on Manifesto quasi-sentences.
Task: left vs right. Held-out Cohen's kappa: <your number>. Seed: 42.
Data: Manifesto Project, pulled under the author's own key, not redistributed.
```

## Where next

You trained, validated, and shipped a model. The stretch notebook shows the same job with the
full Hugging Face Trainer and a parameter-efficient LoRA adapter, and points at the generative
cousin of what you built.